> EDA

- Eksik değerler    →  model hata verir
- Aykırı değerler   →  model yanılır
- Yanlış dtype      →  hesaplama yapılamaz
- Dengesiz dağılım  →  model bias'lanır



1. Duplicate sil          → temiz veriyle başla
2. Eksik değerleri doldur → sonra analiz et
3. Aykırı değer kontrol   → describe() + boxplot
4. Skewness düzelt        → log1p
5. Korelasyon bak         → feature kararı ver



2. %35 eksik — doğru, silme
- Ama "median ile doldur" her zaman doğru değil. Karar ağacı şöyle:
- %35 eksik
- ├── Sayısal mı?
- │   ├── Normal dağılım → mean ile doldur
- │   └── Çarpık / aykırı değer var → median ile doldur ✅
- └── Kategorik mi? → mode ile doldur

- %60+ eksik → sütunu tamamen sil


EDA Workflow — 5 Adım:
1. Veriyi tanı          →  shape, dtypes, missing values
2. Univariate analiz    →  her sütunu ayrı ayrı incele
3. Bivariate analiz     →  iki sütun arasındaki ilişki
4. Multivariate analiz  →  korelasyon, pairplot
5. Özet & karar         →  ne temizlenecek, ne dönüştürülecek

In [1]:
# Adım 1 — Veriyi Tanı
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('../data_analysis/sources/Movies.csv')

# Temel bilgi
print(df.shape)        # kaç satır, kaç sütun
print(df.dtypes)       # her sütunun tipi
print(df.head())       # ilk 5 satır
print(df.describe())   # istatistiksel özet

# Eksik değerler
print(df.isnull().sum())                        # her sütunda kaç eksik
print(df.isnull().sum() / len(df) * 100)        # yüzde olarak

# Duplicate satırlar
print(df.duplicated().sum())                    # kaç tekrar var
df = df.drop_duplicates()                       # temizle

(5275, 7)
Title             str
Year            int64
Genre             str
Duration        int64
Director          str
Rating        float64
Popularity    float64
dtype: object
                                          Title  Year   Genre  Duration  \
0                                   What Is It?  2005   Drama        72   
1                                       Glitter  2001   Drama       104   
2                         The Attic Expeditions  2001  Comedy       100   
3                               Men in Black II  2002  Action        88   
4  Star Wars: Episode II - Attack of the Clones  2002  Action       142   

             Director  Rating  Popularity  
0      Crispin Glover     5.6       21.83  
1  Vondie Curtis-Hall     2.2       81.69  
2       Jeremy Kasten     5.0       42.08  
3    Barry Sonnenfeld     6.2       98.60  
4        George Lucas     6.5       99.58  
              Year     Duration       Rating   Popularity
count  5275.000000  5275.000000  5275.000000  527

> describe() çıktısını nasıl okursun:
-         fiyat      metrekare
- count    1000        1000       ← kaç değer var (eksik yoksa = shape)
- mean     285000      112        ← ortalama
- std      95000       45         ← ne kadar dağınık
- min      80000       30         ← en küçük
- 25%      210000      80         ← Q1
- 50%      275000      105        ← median
- 75%      340000      140        ← Q3
- max      950000      320        ← en büyük → 950000 şüpheli mi?

> Adım 2 — Univariate Analiz
>> Her sütunu tek başına incele.
>>> Sayısal sütunlar için:

In [ ]:
# Histogram — dağılım nasıl?
df['fiyat'].hist(bins=30, figsize=(8,4))
plt.title('Fiyat Dağılımı')
plt.show()

# Boxplot — aykırı değer var mı?
sns.boxplot(x=df['fiyat'])
plt.show()

# Skewness kontrol
print(f"Skew: {df['fiyat'].skew():.2f}")
# > 1 ise log dönüşümü uygula

>>> Kategorik sütunlar için:

In [ ]:
# Value counts — kaç unique değer, dağılım nasıl?
print(df['kategori'].value_counts())

# Bar plot
df['kategori'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
# Boxplot nasıl okunur:

#         │
#    ─────┤  ← max (aykırı değer değil)
#         │
#    ┌────┤  ← Q3 (75%)
#    │    │
#    │────│  ← median (50%)
#    │    │
#    └────┤  ← Q1 (25%)
#         │
#    ─────┤  ← min (aykırı değer değil)
   
#    •    ← aykırı değer (IQR dışında)

>Adım 3 — Bivariate Analiz
>>İki sütun arasındaki ilişkiyi incele.
>>>Sayısal & Sayısal:

In [ ]:
# Scatter plot — doğrusal ilişki var mı?
sns.scatterplot(x='metrekare', y='fiyat', data=df)
plt.show()

# Korelasyon değeri
print(df['metrekare'].corr(df['fiyat']))

>>> Kategorik & Sayısal:

In [ ]:
# Her kategorideki fiyat dağılımı
sns.boxplot(x='kategori', y='fiyat', data=df)
plt.show()

# Gruplar arası anlamlı fark var mı? → ANOVA
# (Az önce öğrendik!)

>>> Kategorik & Kategorik:

In [ ]:
# Crosstab
pd.crosstab(df['sehir'], df['kategori'])

> Adım 4 — Multivariate Analiz

In [ ]:
# Korelasyon heatmap — tüm sayısal sütunlar arası
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), 
            annot=True, 
            cmap='coolwarm',
            fmt='.2f',
            vmin=-1, vmax=1)
plt.title('Korelasyon Matrisi')
plt.show()

# Pairplot — tüm ikili ilişkiler tek seferde
sns.pairplot(df[['fiyat', 'metrekare', 'oda_sayisi', 'bina_yasi']])
plt.show()

>>Pairplot nasıl okunur:

In [ ]:
# Diagonal    →  her sütunun kendi dağılımı (histogram)
# Diğerleri   →  iki sütun arası scatter plot

> Adım 5 — Aykırı Değer Tespiti & Karar

In [ ]:
# IQR yöntemi — az önce öğrendik
def outlier_tespit(df, kolon):
    Q1 = df[kolon].quantile(0.25)
    Q3 = df[kolon].quantile(0.75)
    IQR = Q3 - Q1
    alt = Q1 - 1.5 * IQR
    ust = Q3 + 1.5 * IQR
    
    outlier_sayisi = df[(df[kolon] < alt) | (df[kolon] > ust)].shape[0]
    print(f"{kolon}: {outlier_sayisi} aykırı değer")
    print(f"  Alt sınır: {alt:.2f}, Üst sınır: {ust:.2f}")

for kolon in ['fiyat', 'metrekare', 'bina_yasi']:
    outlier_tespit(df, kolon)

In [ ]:
# Aykırı değerde ne yaparsın — karar ağacı:
# Aykırı değer tespit edildi
# ├── Veri girişi hatası mı?  →  Düzelt veya sil
# ├── Gerçek ama nadir mi?    →  Koru (fraud detection'da değerli)
# └── Modeli bozuyor mu?      →  Cap uygula (winsorize)
#     df['fiyat'] = df['fiyat'].clip(upper=ust_sinir)

In [ ]:
# Tam EDA Template — Her Projede Kullan:

def hizli_eda(df):
    print("="*50)
    print(f"SHAPE: {df.shape}")
    print(f"DUPLICATE: {df.duplicated().sum()}")
    
    print("\n--- EKSİK DEĞERLER ---")
    eksik = df.isnull().sum()
    print(eksik[eksik > 0])
    
    print("\n--- SAYISAL ÖZET ---")
    print(df.describe().round(2))
    
    print("\n--- SKEWNESS ---")
    numerik = df.select_dtypes(include=np.number)
    for col in numerik.columns:
        skew = df[col].skew()
        if abs(skew) > 1:
            print(f"{col}: {skew:.2f} ← dönüşüm gerekebilir")
    
    print("\n--- KATEGORİK SÜTUNLAR ---")
    kategorik = df.select_dtypes(include='object')
    for col in kategorik.columns:
        print(f"{col}: {df[col].nunique()} unique değer")

hizli_eda(df)

In [ ]:
# Sağa çarpık veri → ne uygulardık?
# df['gelir_log'] = np.log1p(df['gelir'])
# Skew=2.8 → çok sağa çarpık → np.log1p uygula → skew 0'a yaklaşır.
# Önce:  [500, 800, 1200, 50000]  → skew=2.8
# Sonra: [6.2, 6.7,  7.1,  10.8] → skew≈0.3
# Bağlantıyı kuramamıştın çünkü ayrı konular gibi görünüyordu — ama EDA'da öğrendiğin her şey birbirine bağlı:
# skew > 1 tespit et (EDA)  →  np.log1p uygula (numpy)  →  modele ver (Faz 2)

In [ ]:
# Zayıf korelasyon = "modele verme" demek değil. Şu soruyu sor:
# Bu feature başka bir feature ile birleşince anlam kazanır mı?
# → yas + gelir → "genç ve yüksek gelirli" → güçlü segment
# Buna feature interaction denir